# 01. QuartzNet 10x4 — обучение char-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: QuartzNet-10x4 (TCS-свёрточный энкодер, ~4M параметров).
HP Random Search оборачивает весь цикл обучения.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Локально: если репо уже установлен (uv sync / pip install -e .), эта ячейка — no-op.

# --- Colab ---
# !git clone --depth 1 https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git
# import sys
# sys.path.insert(0, "ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

# --- Kaggle ---
# import sys
# sys.path.insert(0, "/kaggle/working/ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

print("Deps cell — fill in your platform block above if on Colab/Kaggle.")

## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [ ]:
from pathlib import Path
import torch

# Fill in before running.
TRAIN_ROOT = Path("FILL_ME_IN")       # dir with train audio files (.wav / .mp3)
DEV_ROOT = Path("FILL_ME_IN")         # dir with dev audio files
TEST_ROOT: Path | None = None         # optional, set to Path("...") if have test data
TRAIN_CSV = Path("FILL_ME_IN")        # Kaggle-style CSV: filename, transcription, spk_id, gender, ext, samplerate
DEV_CSV = Path("FILL_ME_IN")
CKPT_ROOT = Path("FILL_ME_IN")        # where best.pt + meta.json will be saved

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

## Шаг 1. Импорты

In [ ]:
import json
import random
import time

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.text.vocab import CharVocab
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig

## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [ ]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 50,
    "grad_accum": 2,
    "subsample_factor": 2,
}
HP_GRID = {
    "lr":                        [0.015, 0.010, 0.005],
    "weight_decay":              [1e-3, 1e-4],
    "dropout":                   [0.05, 0.1, 0.15],
    "d_model":                   [256],
    "warmup_steps":              [3000, 5000, 8000],
    "specaug_freq_mask_param":   [10, 15, 20],
    "specaug_time_mask_param":   [20, 25, 35],
    "speed_prob":                [0.5, 1.0],
    "pitch_prob":                [0.0, 0.3],
    "gain_prob":                 [0.3, 0.7],
    "noise_prob":                [0.0, 0.3],
}
N_TRIALS = 8
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


## Шаг 3. Загрузка записей из CSV

In [ ]:
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")


## Шаг 4. Vocabulary

In [ ]:
vocab = CharVocab()
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [ ]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)

## Шаг 5. HP Random Search — Training Loop

In [ ]:
SEED = 42
random.seed(SEED)
trial_results = []
run_root = CKPT_ROOT / f"01_quartznet_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        gain_prob=hp["gain_prob"],
        noise_prob=hp["noise_prob"],
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=2,
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=5,
        time_mask_max_ratio=0.05,
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
    )

    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_novograd(
        model.parameters(),
        lr=hp["lr"],
        betas=(0.95, 0.5),
        weight_decay=hp["weight_decay"],
    )
    total_steps = len(train_loader) * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
        min_lr_ratio=0.01,
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        val_every_n_epochs=1,
        early_stop_patience=8,
        early_stop_metric="max_speaker_cer",
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "quartznet_10x4",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_val_cer"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

print("\nHP search complete.")


## Шаг 6. Сводный отчёт трайлов

In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_val_cer"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'d_model':>8}")
print("-" * 50)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_val_cer']:>12.4f}"
        f" {hp['lr']:>8.4f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
    )

## Шаг 7. Валидация лучшей модели на dev + greedy predictions

In [ ]:
best_result = trial_results_sorted[0]
best_ckpt = best_result["best_ckpt_path"]
best_hp = best_result["hp"]
print("Best trial val_cer:", best_result["best_val_cer"], "ckpt:", best_ckpt)

model = QuartzNet10x4(
    vocab_size=vocab.vocab_size,
    d_model=best_hp["d_model"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
meta = load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records, vocab=vocab, augmenter=None, target_samplerate=FIXED["samplerate"]
)
dev_loader_eval = DataLoader(
    dev_ds_eval, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2
)

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(batch.transcriptions)
        spks.extend(batch.spk_ids)

digit_preds = [words_to_digits(h) for h in predictions]
cer = compute_cer(refs, digit_preds)
per_spk = compute_per_speaker_cer(refs, digit_preds, spks)
print(f"Dev CER (greedy): {cer:.4f}")
print("Per-speaker CER:", per_spk)

## Шаг 8 (опционально). Beam search + KenLM rescore

In [ ]:
# Optional — requires kenlm + pyctcdecode and a trained LM model.
# See notebooks/experiments/06_kenlm_beam_rescore.ipynb for the full pipeline.
# from gp1.decoding.beam_pyctc import BeamSearchDecoder, BeamSearchConfig


## Шаг 9. Submission (если test данные доступны)

In [ ]:
if TEST_ROOT is not None:
    from gp1.submit.inference_utils import build_test_dataloader, write_submission

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    # Pair filenames (preserved order from SequentialSampler) with predictions.
    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = run_root / "submission.csv"
    write_submission(pairs, submission_path)
    print("Submission saved:", submission_path)
else:
    print("No test_root — skipping submission.")
